In [0]:
import json
from pathlib import Path

# Find the existing state file in your Git folder
repo_state = Path.cwd().parent / "state.json"

print("Current working directory:", Path.cwd())
print("Looking for:", repo_state)
print("File exists:", repo_state.exists())

In [0]:
import shutil
from pathlib import Path

repo_state = Path.cwd().parent / "state.json"
volume_state = Path("/Volumes/workspace/nws/raw/state.json")

# Copy only if a state file doesn't already exist in the Volume
if volume_state.exists():
    print("A state.json already exists in the Volume. No changes made.")
else:
    shutil.copy2(repo_state, volume_state)
    print("State copied successfully.")

print("Volume state exists:", volume_state.exists())

In [0]:
import json
from pathlib import Path

state_path = Path("/Volumes/workspace/nws/raw/state.json")
state = json.loads(state_path.read_text())

print("Observation watermark:", state.get("obs_watermark"))
print("Saved alert IDs:", len(state.get("seen_alerts", [])))
print("Saved stations:", len(state.get("stations", [])))
print("Cached grids:", len(state.get("grids", {})))

In [0]:
import requests

response = requests.get(
    "https://api.weather.gov/alerts/active?area=KS",
    headers={
        "User-Agent": "(NWS-Medallion-Student-Project, l227451@lhr.nu.edu.pk)",
        "Accept": "application/geo+json",
    },
    timeout=20,
)

print("HTTP status:", response.status_code)
print("Alerts returned:", len(response.json().get("features", []))
      if response.status_code == 200 else "Request failed")

In [0]:
import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.insert(0, str(repo_root))

import collect_nws
collect_nws.main("full")

In [0]:
from pathlib import Path
import json

raw_base = Path("/Volumes/workspace/nws/raw")

for endpoint in ["alerts", "forecasts", "observations"]:
    folder = raw_base / "full" / endpoint
    files = sorted(folder.glob("*.json"))

    print(f"\n{endpoint.upper()}: {len(files)} file(s)")

    if files:
        with files[-1].open() as f:
            data = json.load(f)

        if endpoint == "alerts":
            print("Active alerts:", len(data.get("features", [])))
        elif endpoint == "forecasts":
            print("Locations:", len(data.get("grids", {})))
        else:
            print("Stations:", len(data.get("stations", {})))